# lets_route — `route_f` Playground

This notebook runs on the **`route_f`** kernel (Python 3.12, uv-managed).
The same environment is used in the terminal via:

```bash
source .route_f/bin/activate
```

> If the kernel isn't listed in VS Code yet, finish the setup first:
> `./setup_route_f.sh` (or verify with `jupyter kernelspec list`).

In [1]:
import sys, platform
print(f"Python : {sys.version.split()[0]} (executable: {sys.executable})")
print(f"Platform: {platform.platform()}")

try:
    import torch
    print(f"torch : {torch.__version__}")
    print(f"CUDA available : {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"GPU : {torch.cuda.get_device_name(0)}")
    try:
        torch.zeros(1, device=torch.cuda.is_available() and 'cuda' or 'cpu')
        print("tensor allocation ok")
    except Exception as e:
        print("tensor allocation failed:", e)
except ImportError:
    print("torch not installed yet — run ./setup_route_f.sh")

Python : 3.12.13 (executable: /home/mr/Desktop/Fireworks/.route_f/bin/python)
Platform: Linux-7.0.0-38-generic-x86_64-with-glibc2.39
torch : 2.15.0+cu132
CUDA available : True
GPU : NVIDIA GeForce RTX 3060 Laptop GPU
tensor allocation ok


In [2]:
# Quick sanity check of the training stack
import importlib
mods = ["transformers", "datasets", "accelerate", "peft", "pandas", "numpy", "pyarrow"]
for m in mods:
    try:
        mod = importlib.import_module(m)
        print(f"{m:16s} {getattr(mod, '__version__', '?')}")
    except ImportError:
        print(f"{m:16s} NOT INSTALLED")

/home/mr/Desktop/Fireworks/.route_f/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


transformers     5.22.1
datasets         5.1.0
accelerate       1.15.0
peft             0.21.2
pandas           3.1.0
numpy            2.5.4
pyarrow          26.0.0


## Dataset: xRouteBench (`ulab-ai/xRouteBench`)

LLM-routing benchmark — each row is a `(query, candidate model)` pair with the
model's response, token usage, latency, and a `performance` score. A router
learns which model to pick per query, trading off performance vs price.

- **Configs (scenarios):** `llmrouter_generic`, `memory_locomo`, `memory_longmemeval`,
  `timeseries`, `video`, `multimodal_geometry3k`, `multimodal_mathvista`, `personalized`.
- **Raw queries:** `<scenario>_queries` (train/valid/test) — no model executions.
- **Pricing pool:** `llm_candidates` (18 models with prices).
- **No bulk download:** the cells below inspect everything via the HF API and
  stream only a few rows (parquet pulled lazily). Set upstream=False`` only when
  you actually want to download the full dataset for training.

In [3]:
import sys
sys.path.insert(0, ".")
from data.load_xroutebench import list_configs, schema, peek

print("Available configs (from HF datasets-server API, no download):")
for c in list_configs():
    print("  -", c)

cols, samples = schema("llmrouter_generic", "train")
print("\nSchema for llmrouter_generic/train:")
print(cols)

print("\nFirst 2 rows via STREAMING (only the first parquet chunk is read):")
for row in peek("llmrouter_generic", "train", n=2):
    print("  -", {k: row[k] for k in ("task_name", "model_name", "metric")})

Available configs (from HF datasets-server API, no download):
  - llm_candidates
  - llmrouter_generic
  - llmrouter_generic_queries
  - memory_locomo
  - memory_locomo_queries
  - memory_longmemeval
  - memory_longmemeval_queries
  - multimodal_geometry3k
  - multimodal_geometry3k_queries
  - multimodal_mathvista
  - multimodal_mathvista_queries
  - personalized
  - personalized_queries
  - timeseries
  - timeseries_queries
  - video
  - video_queries

Schema for llmrouter_generic/train:
['task_name', 'query', 'ground_truth', 'metric', 'choices', 'task_id', 'model_name', 'response', 'token_num', 'input_tokens', 'output_tokens', 'response_time', 'performance', 'embedding_id']

First 2 rows via STREAMING (only the first parquet chunk is read):
  - {'task_name': 'mbpp', 'model_name': 'qwen2.5-7b-instruct', 'metric': 'code_eval'}
  - {'task_name': 'mbpp', 'model_name': 'gemma-2-9b-it', 'metric': 'code_eval'}


In [ ]:
# Download the pinned official Qwen3.5-0.8B checkpoint into this repo.
from pathlib import Path
from huggingface_hub import snapshot_download

MODEL_ID = "Qwen/Qwen3.5-0.8B"
MODEL_REVISION = "2fc06364715b967f1860aea9cf38778875588b17"
repo_root = next(
    path for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "route_f_playground.ipynb").exists()
)
MODEL_DIR = repo_root / "models" / "qwen3.5-0.8b"

snapshot_path = snapshot_download(
    repo_id=MODEL_ID,
    revision=MODEL_REVISION,
    local_dir=MODEL_DIR,
)
downloaded_bytes = sum(
    path.stat().st_size for path in MODEL_DIR.rglob("*") if path.is_file()
)
print(f"Downloaded {MODEL_ID}@{MODEL_REVISION[:8]}")
print(f"Location: {snapshot_path}")
print(f"Size: {downloaded_bytes / 1024**3:.2f} GiB")